# Robust RBAC Decorator with Custom Exceptions

This exercise builds on our solution from the RBAC Decorator from the section Generators and Decorators.

You have already created a basic RBAC decorator factory. Now, you need to make it robust and production-ready. A production system cannot assume that inputs will always be correct. Your decorator must validate its inputs and provide clear, domain-specific errors when something goes wrong.

Your task is to enhance the `require_role` decorator factory by adding input validation and a custom exception for authorization failures.

## Functional Requirements:

### Define a Custom Exception:

Create a new custom exception class named `AuthorizationError` that inherits from `Exception`.

- Its `__init__` method should accept a `user_name` and a `required_role`.
- It should store these as attributes, and generate a clear error message such as `"User 'alice' lacks the required role: 'admin'."`

### Enhance the Decorator:

Modify the `require_role` decorator's wrapper to perform thorough validation before checking permissions.

- **Validation Step 1:** If the `user` keyword argument is missing, raise a `ValueError`.
- **Validation Step 2:** If the `user` object is not a dictionary, or if it does not contain a `roles` key, or if the `roles` value is not a list, raise a `ValueError`.
- **Permission Check:** If validation passes, but the user lacks the required role, you must now raise your new `AuthorizationError` instead of the generic `PermissionError`. Pass the user's name and the required role to its constructor.

## Example:

```python
# Custom exception for clarity
class AuthorizationError(Exception):
    # ... implementation ...
    pass

@require_role('admin')
def restart_server(*, user, server_id):
    """Restarts a server after a permission check."""
    return f"Server {server_id} restart initiated."

admin_user = {'name': 'alice', 'roles': ['admin']}
viewer_user = {'name': 'bob', 'roles': ['viewer']}
malformed_user = {'name': 'eve'} # Missing 'roles' key

# Succeeds as before
restart_server(user=admin_user, server_id='web-01')

# Now raises AuthorizationError('bob', 'admin')
# restart_server(user=viewer_user, server_id='db-01')

# Now raises ValueError due to malformed input
# restart_server(user=malformed_user, server_id='app-01')
```

## How Your Solution Will Be Tested:

All tests from the previous exercise should still pass (with `AuthorizationError` replacing `PermissionError`).

New tests will be added to verify the validation logic:

- A call with a missing `user` keyword argument must raise `ValueError`.
- A call where `user` is not a dictionary must raise `TypeError`.
- A call where `user` is missing the `roles` key must raise `ValueError`.
- A call where `user['roles']` is not a list must raise `ValueError`.
- A failed permission check must raise `AuthorizationError` and the exception object must contain the correct `user_name` and `required_role`.

In [ ]:
import functools

# TODO: Define the AuthorizationError custom exception.
# It should inherit from Exception.
# Its __init__ method should accept `user_name` and `required_role`.
# It should store these as attributes and create a descriptive error message to pass to the parent class's __init__.

def require_role(required_role):
    """
    A decorator factory that creates a decorator to check for a specific user role, with robust validation and custom exceptions.
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # This is the solution from our previous exercise.
            # You will modify it.
            # Start with this known-good implementation.

            # user = kwargs.get('user')
            # user_roles = user.get('roles', [])
            # if required_role not in user_roles:
            #     raise PermissionError(f"User lacks the required role: '{required_role}'.")
            # return func(*args, **kwargs)

            # TODO: Check if the 'user' keyword argument exists in kwargs and raise a ValueError if not.
            # TODO: Raise a ValueError if the user keyword argument is not a dict, is missing a 'roles' key, or if 'roles' is not a list.
            # TODO: Modify the permission check to raise an AuthorizationError instead of PermissionError.
            pass
        return wrapper
    return decorator

In [ ]:
import functools


class AuthorizationError(Exception):
    """
    Custom exception raised when a user does not have
    the required role.
    """

    def __init__(self, user_name, required_role):
        self.user_name = user_name
        self.required_role = required_role

        message = (
            f"User '{user_name}' lacks the required role: "
            f"'{required_role}'."
        )
        super().__init__(message)


def require_role(required_role):
    """
    A decorator factory that creates a decorator to check
    for a specific user role, with robust validation and
    custom exceptions.
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):

            # Validation 1: user keyword argument must exist
            if 'user' not in kwargs:
                raise ValueError("Missing required keyword argument: 'user'.")

            user = kwargs['user']

            # Validation 2: user must be a dictionary
            if not isinstance(user, dict):
                raise TypeError("'user' must be a dictionary.")

            # Validation 3: user must contain 'roles'
            if 'roles' not in user:
                raise ValueError("User dictionary must contain a 'roles' key.")

            # Validation 4: roles must be a list
            if not isinstance(user['roles'], list):
                raise ValueError("'roles' must be a list.")

            # Permission check
            if required_role not in user['roles']:
                user_name = user.get('name', 'Unknown')
                raise AuthorizationError(user_name, required_role)

            return func(*args, **kwargs)

        return wrapper

    return decorator